# RQ4 -- Scenario 3 -- Prompt + Net

Prompt-engineered hub-region prompt (ADHD domain knowledge) PLUS Yeo-7 functional network membership per hub region.

This notebook covers all **5 cases** of this scenario: Clinical K5, Clinical K10, Random K5, Random K10, No Perturb,
across all **6 models** (Llama-3.1-8B-Instruct, DeepSeek-R1-Distill-Qwen-14B, Qwen2.5-14B-Instruct, Qwen3-14B, GPT-Luna, DeepSeek-V4-Flash).


In [2]:
import sys

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["DGLBACKEND"] = "pytorch"

import torch

DEVICE = torch.device("cuda:0")  # GPU 1 becomes cuda:0 now

print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_capability(0))


NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition
(12, 0)


## 1. Model selection -- Local (Hugging Face) models

Loading mechanism (4-bit bnb quantization + `AutoModelForCausalLM`) is identical for all 4 local models; only `MODEL_ID` changes. Exactly one line below should be uncommented.

If the active model for this notebook is an **API** model (GPT-Luna / DeepSeek-V4), this whole cell is inert -- skip to section 2.

In [3]:

import torch
from collections import defaultdict
import statistics as stats
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()
token = os.getenv("HF_TOKEN")
login(token=token)

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"   # LLAMA (meta-llama/Llama-3.1-8B-Instruct)
# MODEL_ID = "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B"   # DEEPSEEK-R1-DISTILL-QWEN-14B (deepseek-ai/DeepSeek-R1-Distill-Qwen-14B)
# MODEL_ID = "Qwen/Qwen2.5-14B-Instruct"   # QWEN2.5-14B-INSTRUCT (Qwen/Qwen2.5-14B-Instruct)
# MODEL_ID = "Qwen/Qwen3-14B"   # QWEN3-14B (Qwen/Qwen3-14B)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",       # spreads layers across available GPU(s), offloads to CPU if needed
    torch_dtype=torch.bfloat16,
)
model.eval()
print("Model loaded.")


/home/siu856622573/.conda/envs/py10_roy/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights:   1%|          | 2/291 [00:00<00:25, 11.25it/s]/home/siu856622573/.conda/envs/py10_roy/lib/python3.10/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 291/291 [00:01<00:00, 186.28it/s]


Model loaded.


## 2. Model selection -- API-based models

GPT-Luna and DeepSeek-V4-Flash are served behind an OpenAI-compatible endpoint instead of being loaded locally. Uncomment exactly one full block below **and** comment out section 1 if you switch to one of these.

If the active model for this notebook is a **local HF** model, this whole cell is inert.

In [4]:
# # --- MODEL: GPT-LUNA (GPT-5.6-LUNA, API) ---
# import os
# from dotenv import load_dotenv
# from openai import OpenAI

# load_dotenv()

# api_key = os.getenv("GPT_LUNA")

# endpoint = os.getenv("END_POINT")
# MODEL_ID = "GPT-5.6-LUNA"

# client = OpenAI(
#     base_url=f"{endpoint}/openai/v1/",
#     api_key=api_key,
# )

# response = client.chat.completions.create(
#     model=MODEL_ID,
#     messages=[
#         {
#             "role": "user",
#             "content": "What is the capital of France?"
#         }
#     ],
# )

# print(response.choices[0].message.content)

# # --- MODEL: DEEPSEEK-V4-FLASH (DeepSeek-V4-Flash, API) ---
# import os
# from dotenv import load_dotenv
# from openai import OpenAI

# load_dotenv()

# api_key = os.getenv("DEEPSEEK_API")

# endpoint = os.getenv("END_POINT")
# MODEL_ID = "DeepSeek-V4-Flash"

# client = OpenAI(
#     base_url=f"{endpoint}/openai/v1/",
#     api_key=api_key,
# )

# response = client.chat.completions.create(
#     model=MODEL_ID,
#     messages=[
#         {
#             "role": "user",
#             "content": "What is the capital of France?"
#         }
#     ],
# )

# print(response.choices[0].message.content)


## 3. Load files -- shared constants (unchanged across models/cases)

In [5]:
import os
import json
import re
import torch
import pandas as pd

RQ1_DIR = "/home/siu856622573/Code/RQ_1"
CODE_DIR = "/home/siu856622573/Code"

EDGE_LIST_NUMBERS = "edge_list_top10pos_numbers.json"   # {subject_id: "Node feature:\n...\n\nEdge feature:\n..."}
EDGE_LIST_NAMES = "edge_list_top10pos_names.json"
ZSCORED_NUMBERS = "zscore_top10pos_numbers.json"
ZSCORED_NAMES = "zscore_top10pos_names.json"
ROI_NAMES_JSON = os.path.join(CODE_DIR, "aal_roi_names.json")           # region-name legend
META_CSV = os.path.join(CODE_DIR, "subject_summary.csv")
DATASET_NAME = "ADHD-200"
LABEL1, LABEL2 = "Control", "ADHD"
EDGE_LIST_NAMES_ONLY = "edge_only_from_top10pos_names.json"

HUB_GROUND_TRUTH = os.path.join(CODE_DIR, "hub_ground_truth_20roi.json")   # {subject_id: "Control" or "ADHD"}, no perturbation, no network
COMMUNITY_GROUND_TRUTH = "community_ground_truth.json"

CLINICAL_HUB_PERTURBED_K5  = os.path.join(RQ1_DIR, "inputs", "hub_clinical_perturbed_k5.json")
CLINICAL_HUB_PERTURBED_K10 = os.path.join(RQ1_DIR, "inputs", "hub_clinical_perturbed_k10.json")
RANDOM_HUB_PERTURBED_K5    = os.path.join(RQ1_DIR, "inputs", "hub_random_perturbed_k5.json")
RANDOM_HUB_PERTURBED_K10   = os.path.join(RQ1_DIR, "inputs", "hub_random_perturbed_k10.json")

HUB_GROUND_TRUTH_NETWORK          = os.path.join(RQ1_DIR, "inputs", "hub_not_perturbed_wnetwork.json")          # no perturbation, WITH network attribute (scenario 3)
HUB_CLINICAL_PERTURBED_NETWORK_K5  = os.path.join(RQ1_DIR, "inputs", "hub_clinical_perturbed_wnetwork_k5.json")
HUB_CLINICAL_PERTURBED_NETWORK_K10 = os.path.join(RQ1_DIR, "inputs", "hub_clinical_perturbed_wnetwork_k10.json")
HUB_RANDOM_PERTURBED_NETWORK_K5    = os.path.join(RQ1_DIR, "inputs", "hub_random_perturbed_wnetwork_k5.json")
HUB_RANDOM_PERTURBED_NETWORK_K10   = os.path.join(RQ1_DIR, "inputs", "hub_random_perturbed_wnetwork_k10.json")

PREPROCESS_TEMPLATE = "AAL116"
TASK_DESC = "interpreting functional connectivity patterns and reasoning about the most likely diagnostic group"
N_SUBJECTS = 768
MAX_NEW_TOKENS = 800


## 4. Select input file for this case -- Scenario 3 -- Prompt + Net

5 cases available for this scenario: Clinical K5, Clinical K10, Random K5, Random K10, No Perturb. Exactly one `with open(...)` line below should be uncommented.

In [6]:
with open(HUB_CLINICAL_PERTURBED_NETWORK_K5) as f:   # ACTIVE CASE: Clinical perturbation, K=5
# with open(HUB_CLINICAL_PERTURBED_NETWORK_K10) as f:   # CASE: Clinical perturbation, K=10
# with open(HUB_RANDOM_PERTURBED_NETWORK_K5) as f:   # CASE: Random perturbation, K=5
# with open(HUB_RANDOM_PERTURBED_NETWORK_K10) as f:   # CASE: Random perturbation, K=10
# with open(HUB_GROUND_TRUTH_NETWORK) as f:   # CASE: No perturbation
    structure_prompts_all = json.load(f)

if os.path.exists(ROI_NAMES_JSON):
    with open(ROI_NAMES_JSON) as f:
        roi_names = json.load(f)
    print(f"Loaded {len(roi_names)} AAL116 ROI names")
else:
    roi_names = None
    print("[warn] aal_roi_names.json not found -- description will omit the ROI-name legend")

if os.path.exists(META_CSV):
    meta = pd.read_csv(META_CSV, dtype={"subject_id": str})
else:
    meta = None

subject_ids = list(structure_prompts_all.keys())[:N_SUBJECTS]
print(f"Loaded {len(structure_prompts_all)} subjects, running on first {len(subject_ids)}")
brain_graph_texts = dict(structure_prompts_all)
print("\nExample BrainGraph text (subject 1):")
print(brain_graph_texts[subject_ids[0]])


Loaded 116 AAL116 ROI names
Loaded 768 subjects, running on first 768

Example BrainGraph text (subject 1):
[{'roi': 'Parietal_Sup_L', 'degree': 20, 'cohort_z': 2.683022361167184, 'network': 'DorsAttn'}, {'roi': 'Temporal_Pole_Mid_L', 'degree': 19, 'cohort_z': 1.7847711032116718, 'network': 'Limbic'}, {'roi': 'Parietal_Sup_R', 'degree': 17, 'cohort_z': 1.7815097482585058, 'network': 'DorsAttn'}, {'roi': 'Frontal_Sup_Orb_L', 'degree': 20, 'cohort_z': 1.7543198922436116, 'network': 'Limbic'}, {'roi': 'Cerebelum_Crus1_R', 'degree': 20, 'cohort_z': 1.6306481499085175, 'network': 'Vis'}, {'roi': 'Temporal_Pole_Mid_R', 'degree': 17, 'cohort_z': 1.194236484049217, 'network': 'Limbic'}, {'roi': 'Temporal_Sup_L', 'degree': 22, 'cohort_z': 1.07962243332119, 'network': 'SomMot'}, {'roi': 'Heschl_L', 'degree': 18, 'cohort_z': 1.068814200841081, 'network': 'SomMot'}, {'roi': 'Frontal_Sup_Medial_R', 'degree': 17, 'cohort_z': 1.0513686408965555, 'network': 'Default'}, {'roi': 'Insula_L', 'degree': 22

## 5. Prompt template -- Scenario 3 -- Prompt + Net

Fields used: ROI, degree, cohort_z, network (Yeo-7) + ADHD domain-knowledge description. This template is fixed for the whole notebook (it defines the scenario) and identical across all 5 cases and all 6 models.

In [7]:
def build_stage1_prompt(subject_id, brain_graph_text):
    hub_regions_text = "; ".join([
        f"{h['roi']} (degree:{h['degree']} cohort_z:{h['cohort_z']:+.2f} network:{h['network']})"
        for h in brain_graph_text
    ])

    return f"""Template: {DATASET_NAME} dataset, {PREPROCESS_TEMPLATE} atlas, 116 ROIs.
Description: The top 20 positive ROI-degree-based hub regions extracted from the subject's resting-state functional connectivity graph. ADHD has been associated with reduced frontal-striatal hub strength and altered default mode network (DMN) hub connectivity. Each hub includes its ROI, degree (number of positive connections), cohort_z (deviation from the cohort mean; positive = above average, negative = below average), and network (Yeo-7 functional network membership).
Task: Based only on these hub regions, predict whether the subject belongs to {LABEL1} or {LABEL2}, and provide a confidence score (0–1) for both classes.
Request: Your output must strictly follow this JSON structure and contain nothing else:
{{
  "prediction": "{LABEL1} or {LABEL2}",
  "class_confidence": {{
    "{LABEL1}": 0.0,
    "{LABEL2}": 0.0
  }},
  "reasoning": "1-2 sentences"
}}
Hub regions: {hub_regions_text}"""


import tiktoken
try:
    enc = tiktoken.get_encoding("o200k_base")
except Exception:
    enc = tiktoken.get_encoding("cl100k_base")

first_sid = subject_ids[0]
first_prompt = build_stage1_prompt(first_sid, brain_graph_texts[first_sid])  # unchanged
print(f"=== INPUT SANITY CHECK -- Subject {first_sid} ===")
print(first_prompt)
total_input_tokens = len(enc.encode(first_prompt))
print(f"\n--- prompt length: {len(first_prompt)} chars, ~{total_input_tokens} tokens (estimated) ---")


=== INPUT SANITY CHECK -- Subject 1018959 ===
Template: ADHD-200 dataset, AAL116 atlas, 116 ROIs.
Description: The top 20 positive ROI-degree-based hub regions extracted from the subject's resting-state functional connectivity graph. ADHD has been associated with reduced frontal-striatal hub strength and altered default mode network (DMN) hub connectivity. Each hub includes its ROI, degree (number of positive connections), cohort_z (deviation from the cohort mean; positive = above average, negative = below average), and network (Yeo-7 functional network membership).
Task: Based only on these hub regions, predict whether the subject belongs to Control or ADHD, and provide a confidence score (0–1) for both classes.
Request: Your output must strictly follow this JSON structure and contain nothing else:
{
  "prediction": "Control or ADHD",
  "class_confidence": {
    "Control": 0.0,
    "ADHD": 0.0
  },
  "reasoning": "1-2 sentences"
}
Hub regions: Parietal_Sup_L (degree:20 cohort_z:+2.68 

## 6. Output paths -- OUTPUT_DIR / CHECKPOINT_PATH / FINAL_PATH / LOG_PATH

Depends on **both** the active model (section 1/2) and the active case (section 4). Every `MODEL | SCENARIO | PERTURBATION | K` combination for this scenario is listed below; exactly one block should be uncommented, matching whichever model/case you activated above.

In [8]:
# --- LLAMA (meta-llama/Llama-3.1-8B-Instruct) | Scenario 3 -- Prompt + Net | Clinical perturbation, K=5 ---
OUTPUT_DIR = "/home/siu856622573/Code/RQ_1/llama-3.1-8b-instruct"
CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "llama_clinical_prompt+net_k5_checkpoint.json")
FINAL_PATH = os.path.join(OUTPUT_DIR, "llama_clinical_prompt+net_k5.json")
LOG_PATH = os.path.join(OUTPUT_DIR, "llama_clinical_prompt+net_k5_errors.log")

# --- LLAMA (meta-llama/Llama-3.1-8B-Instruct) | Scenario 3 -- Prompt + Net | Clinical perturbation, K=10 ---
# OUTPUT_DIR = "/home/siu856622573/Code/RQ_1/llama-3.1-8b-instruct"
# CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "llama_clinical_prompt+net_k10_checkpoint.json")
# FINAL_PATH = os.path.join(OUTPUT_DIR, "llama_clinical_prompt+net_k10.json")
# LOG_PATH = os.path.join(OUTPUT_DIR, "llama_clinical_prompt+net_k10_errors.log")

# --- LLAMA (meta-llama/Llama-3.1-8B-Instruct) | Scenario 3 -- Prompt + Net | Random perturbation, K=5 ---
# OUTPUT_DIR = "/home/siu856622573/Code/RQ_1/llama-3.1-8b-instruct"
# CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "llama_random_prompt+net_k5_checkpoint.json")
# FINAL_PATH = os.path.join(OUTPUT_DIR, "llama_random_prompt+net_k5.json")
# LOG_PATH = os.path.join(OUTPUT_DIR, "llama_random_prompt+net_k5_errors.log")

# --- LLAMA (meta-llama/Llama-3.1-8B-Instruct) | Scenario 3 -- Prompt + Net | Random perturbation, K=10 ---
# OUTPUT_DIR = "/home/siu856622573/Code/RQ_1/llama-3.1-8b-instruct"
# CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "llama_random_prompt+net_k10_checkpoint.json")
# FINAL_PATH = os.path.join(OUTPUT_DIR, "llama_random_prompt+net_k10.json")
# LOG_PATH = os.path.join(OUTPUT_DIR, "llama_random_prompt+net_k10_errors.log")

# --- LLAMA (meta-llama/Llama-3.1-8B-Instruct) | Scenario 3 -- Prompt + Net | No perturbation ---
# OUTPUT_DIR = "/home/siu856622573/Code/RQ_1/llama-3.1-8b-instruct"
# CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "llama_no_perturb_prompt+net_checkpoint.json")
# FINAL_PATH = os.path.join(OUTPUT_DIR, "llama_no_perturb_prompt+net.json")
# LOG_PATH = os.path.join(OUTPUT_DIR, "llama_no_perturb_prompt+net_errors.log")

# --- DEEPSEEK-R1-DISTILL-QWEN-14B (deepseek-ai/DeepSeek-R1-Distill-Qwen-14B) | Scenario 3 -- Prompt + Net | Clinical perturbation, K=5 ---
# OUTPUT_DIR = "/home/siu856622573/Code/RQ_1/deepseek-r1-distill-qwen-14b"
# CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "deepseek_clinical_prompt+net_k5_checkpoint.json")
# FINAL_PATH = os.path.join(OUTPUT_DIR, "deepseek_clinical_prompt+net_k5.json")
# LOG_PATH = os.path.join(OUTPUT_DIR, "deepseek_clinical_prompt+net_k5_errors.log")

# --- DEEPSEEK-R1-DISTILL-QWEN-14B (deepseek-ai/DeepSeek-R1-Distill-Qwen-14B) | Scenario 3 -- Prompt + Net | Clinical perturbation, K=10 ---
# OUTPUT_DIR = "/home/siu856622573/Code/RQ_1/deepseek-r1-distill-qwen-14b"
# CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "deepseek_clinical_prompt+net_k10_checkpoint.json")
# FINAL_PATH = os.path.join(OUTPUT_DIR, "deepseek_clinical_prompt+net_k10.json")
# LOG_PATH = os.path.join(OUTPUT_DIR, "deepseek_clinical_prompt+net_k10_errors.log")

# --- DEEPSEEK-R1-DISTILL-QWEN-14B (deepseek-ai/DeepSeek-R1-Distill-Qwen-14B) | Scenario 3 -- Prompt + Net | Random perturbation, K=5 ---
# OUTPUT_DIR = "/home/siu856622573/Code/RQ_1/deepseek-r1-distill-qwen-14b"
# CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "deepseek_random_prompt+net_k5_checkpoint.json")
# FINAL_PATH = os.path.join(OUTPUT_DIR, "deepseek_random_prompt+net_k5.json")
# LOG_PATH = os.path.join(OUTPUT_DIR, "deepseek_random_prompt+net_k5_errors.log")

# --- DEEPSEEK-R1-DISTILL-QWEN-14B (deepseek-ai/DeepSeek-R1-Distill-Qwen-14B) | Scenario 3 -- Prompt + Net | Random perturbation, K=10 ---
# OUTPUT_DIR = "/home/siu856622573/Code/RQ_1/deepseek-r1-distill-qwen-14b"
# CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "deepseek_random_prompt+net_k10_checkpoint.json")
# FINAL_PATH = os.path.join(OUTPUT_DIR, "deepseek_random_prompt+net_k10.json")
# LOG_PATH = os.path.join(OUTPUT_DIR, "deepseek_random_prompt+net_k10_errors.log")

# --- DEEPSEEK-R1-DISTILL-QWEN-14B (deepseek-ai/DeepSeek-R1-Distill-Qwen-14B) | Scenario 3 -- Prompt + Net | No perturbation ---
# OUTPUT_DIR = "/home/siu856622573/Code/RQ_1/deepseek-r1-distill-qwen-14b"
# CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "deepseek_no_perturb_prompt+net_checkpoint.json")
# FINAL_PATH = os.path.join(OUTPUT_DIR, "deepseek_no_perturb_prompt+net.json")
# LOG_PATH = os.path.join(OUTPUT_DIR, "deepseek_no_perturb_prompt+net_errors.log")

# --- QWEN2.5-14B-INSTRUCT (Qwen/Qwen2.5-14B-Instruct) | Scenario 3 -- Prompt + Net | Clinical perturbation, K=5 ---
# OUTPUT_DIR = "/home/siu856622573/Code/RQ_1/qwen"
# CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "qwen_clinical_prompt+net_k5_checkpoint.json")
# FINAL_PATH = os.path.join(OUTPUT_DIR, "qwen_clinical_prompt+net_k5.json")
# LOG_PATH = os.path.join(OUTPUT_DIR, "qwen_clinical_prompt+net_k5_errors.log")

# --- QWEN2.5-14B-INSTRUCT (Qwen/Qwen2.5-14B-Instruct) | Scenario 3 -- Prompt + Net | Clinical perturbation, K=10 ---
# OUTPUT_DIR = "/home/siu856622573/Code/RQ_1/qwen"
# CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "qwen_clinical_prompt+net_k10_checkpoint.json")
# FINAL_PATH = os.path.join(OUTPUT_DIR, "qwen_clinical_prompt+net_k10.json")
# LOG_PATH = os.path.join(OUTPUT_DIR, "qwen_clinical_prompt+net_k10_errors.log")

# --- QWEN2.5-14B-INSTRUCT (Qwen/Qwen2.5-14B-Instruct) | Scenario 3 -- Prompt + Net | Random perturbation, K=5 ---
# OUTPUT_DIR = "/home/siu856622573/Code/RQ_1/qwen"
# CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "qwen_random_prompt+net_k5_checkpoint.json")
# FINAL_PATH = os.path.join(OUTPUT_DIR, "qwen_random_prompt+net_k5.json")
# LOG_PATH = os.path.join(OUTPUT_DIR, "qwen_random_prompt+net_k5_errors.log")

# --- QWEN2.5-14B-INSTRUCT (Qwen/Qwen2.5-14B-Instruct) | Scenario 3 -- Prompt + Net | Random perturbation, K=10 ---
# OUTPUT_DIR = "/home/siu856622573/Code/RQ_1/qwen"
# CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "qwen_random_prompt+net_k10_checkpoint.json")
# FINAL_PATH = os.path.join(OUTPUT_DIR, "qwen_random_prompt+net_k10.json")
# LOG_PATH = os.path.join(OUTPUT_DIR, "qwen_random_prompt+net_k10_errors.log")

# --- QWEN2.5-14B-INSTRUCT (Qwen/Qwen2.5-14B-Instruct) | Scenario 3 -- Prompt + Net | No perturbation ---
# OUTPUT_DIR = "/home/siu856622573/Code/RQ_1/qwen"
# CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "qwen_no_perturb_prompt+net_checkpoint.json")
# FINAL_PATH = os.path.join(OUTPUT_DIR, "qwen_no_perturb_prompt+net.json")
# LOG_PATH = os.path.join(OUTPUT_DIR, "qwen_no_perturb_prompt+net_errors.log")

# --- QWEN3-14B (Qwen/Qwen3-14B) | Scenario 3 -- Prompt + Net | Clinical perturbation, K=5 ---
# OUTPUT_DIR = "/home/siu856622573/Code/RQ_1/qwen3_14b"
# CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "qwen3_14b_clinical_prompt+net_k5_checkpoint.json")
# FINAL_PATH = os.path.join(OUTPUT_DIR, "qwen3_14b_clinical_prompt+net_k5.json")
# LOG_PATH = os.path.join(OUTPUT_DIR, "qwen3_14b_clinical_prompt+net_k5_errors.log")

# --- QWEN3-14B (Qwen/Qwen3-14B) | Scenario 3 -- Prompt + Net | Clinical perturbation, K=10 ---
# OUTPUT_DIR = "/home/siu856622573/Code/RQ_1/qwen3_14b"
# CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "qwen3_14b_clinical_prompt+net_k10_checkpoint.json")
# FINAL_PATH = os.path.join(OUTPUT_DIR, "qwen3_14b_clinical_prompt+net_k10.json")
# LOG_PATH = os.path.join(OUTPUT_DIR, "qwen3_14b_clinical_prompt+net_k10_errors.log")

# --- QWEN3-14B (Qwen/Qwen3-14B) | Scenario 3 -- Prompt + Net | Random perturbation, K=5 ---
# OUTPUT_DIR = "/home/siu856622573/Code/RQ_1/qwen3_14b"
# CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "qwen3_14b_random_prompt+net_k5_checkpoint.json")
# FINAL_PATH = os.path.join(OUTPUT_DIR, "qwen3_14b_random_prompt+net_k5.json")
# LOG_PATH = os.path.join(OUTPUT_DIR, "qwen3_14b_random_prompt+net_k5_errors.log")

# --- QWEN3-14B (Qwen/Qwen3-14B) | Scenario 3 -- Prompt + Net | Random perturbation, K=10 ---
# OUTPUT_DIR = "/home/siu856622573/Code/RQ_1/qwen3_14b"
# CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "qwen3_14b_random_prompt+net_k10_checkpoint.json")
# FINAL_PATH = os.path.join(OUTPUT_DIR, "qwen3_14b_random_prompt+net_k10.json")
# LOG_PATH = os.path.join(OUTPUT_DIR, "qwen3_14b_random_prompt+net_k10_errors.log")

# --- QWEN3-14B (Qwen/Qwen3-14B) | Scenario 3 -- Prompt + Net | No perturbation ---
# OUTPUT_DIR = "/home/siu856622573/Code/RQ_1/qwen3_14b"
# CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "qwen3_14b_no_perturb_prompt+net_checkpoint.json")
# FINAL_PATH = os.path.join(OUTPUT_DIR, "qwen3_14b_no_perturb_prompt+net.json")
# LOG_PATH = os.path.join(OUTPUT_DIR, "qwen3_14b_no_perturb_prompt+net_errors.log")

# --- GPT-LUNA (GPT-5.6-LUNA, API) | Scenario 3 -- Prompt + Net | Clinical perturbation, K=5 ---
# OUTPUT_DIR = "/home/siu856622573/Code/RQ_1/gpt_luna"
# CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "gpt_clinical_prompt+net_k5_checkpoint.json")
# FINAL_PATH = os.path.join(OUTPUT_DIR, "gpt_clinical_prompt+net_k5.json")
# LOG_PATH = os.path.join(OUTPUT_DIR, "gpt_clinical_prompt+net_k5_errors.log")

# --- GPT-LUNA (GPT-5.6-LUNA, API) | Scenario 3 -- Prompt + Net | Clinical perturbation, K=10 ---
# OUTPUT_DIR = "/home/siu856622573/Code/RQ_1/gpt_luna"
# CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "gpt_clinical_prompt+net_k10_checkpoint.json")
# FINAL_PATH = os.path.join(OUTPUT_DIR, "gpt_clinical_prompt+net_k10.json")
# LOG_PATH = os.path.join(OUTPUT_DIR, "gpt_clinical_prompt+net_k10_errors.log")

# --- GPT-LUNA (GPT-5.6-LUNA, API) | Scenario 3 -- Prompt + Net | Random perturbation, K=5 ---
# OUTPUT_DIR = "/home/siu856622573/Code/RQ_1/gpt_luna"
# CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "gpt_random_prompt+net_k5_checkpoint.json")
# FINAL_PATH = os.path.join(OUTPUT_DIR, "gpt_random_prompt+net_k5.json")
# LOG_PATH = os.path.join(OUTPUT_DIR, "gpt_random_prompt+net_k5_errors.log")

# --- GPT-LUNA (GPT-5.6-LUNA, API) | Scenario 3 -- Prompt + Net | Random perturbation, K=10 ---
# OUTPUT_DIR = "/home/siu856622573/Code/RQ_1/gpt_luna"
# CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "gpt_random_prompt+net_k10_checkpoint.json")
# FINAL_PATH = os.path.join(OUTPUT_DIR, "gpt_random_prompt+net_k10.json")
# LOG_PATH = os.path.join(OUTPUT_DIR, "gpt_random_prompt+net_k10_errors.log")

# --- GPT-LUNA (GPT-5.6-LUNA, API) | Scenario 3 -- Prompt + Net | No perturbation ---
# OUTPUT_DIR = "/home/siu856622573/Code/RQ_1/gpt_luna"
# CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "gpt_no_perturb_prompt+net_scn3_checkpoint.json")
# FINAL_PATH = os.path.join(OUTPUT_DIR, "gpt_no_perturb_prompt+net_scn3.json")
# LOG_PATH = os.path.join(OUTPUT_DIR, "gpt_no_perturb_prompt+net_scn3_errors.log")

# --- DEEPSEEK-V4-FLASH (DeepSeek-V4-Flash, API) | Scenario 3 -- Prompt + Net | Clinical perturbation, K=5 ---
# OUTPUT_DIR = "/home/siu856622573/Code/RQ_1/deepseek_v4"
# CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "deepseek_v4_clinical_prompt+net_k5_checkpoint.json")
# FINAL_PATH = os.path.join(OUTPUT_DIR, "deepseek_v4_clinical_prompt+net_k5.json")
# LOG_PATH = os.path.join(OUTPUT_DIR, "deepseek_v4_clinical_prompt+net_k5_errors.log")

# --- DEEPSEEK-V4-FLASH (DeepSeek-V4-Flash, API) | Scenario 3 -- Prompt + Net | Clinical perturbation, K=10 ---
# OUTPUT_DIR = "/home/siu856622573/Code/RQ_1/deepseek_v4"
# CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "deepseek_v4_clinical_prompt+net_k10_checkpoint.json")
# FINAL_PATH = os.path.join(OUTPUT_DIR, "deepseek_v4_clinical_prompt+net_k10.json")
# LOG_PATH = os.path.join(OUTPUT_DIR, "deepseek_v4_clinical_prompt+net_k10_errors.log")

# --- DEEPSEEK-V4-FLASH (DeepSeek-V4-Flash, API) | Scenario 3 -- Prompt + Net | Random perturbation, K=5 ---
# OUTPUT_DIR = "/home/siu856622573/Code/RQ_1/deepseek_v4"
# CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "deepseek_v4_random_prompt+net_k5_checkpoint.json")
# FINAL_PATH = os.path.join(OUTPUT_DIR, "deepseek_v4_random_prompt+net_k5.json")
# LOG_PATH = os.path.join(OUTPUT_DIR, "deepseek_v4_random_prompt+net_k5_errors.log")

# --- DEEPSEEK-V4-FLASH (DeepSeek-V4-Flash, API) | Scenario 3 -- Prompt + Net | Random perturbation, K=10 ---
# OUTPUT_DIR = "/home/siu856622573/Code/RQ_1/deepseek_v4"
# CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "deepseek_v4_random_prompt+net_k10_checkpoint.json")
# FINAL_PATH = os.path.join(OUTPUT_DIR, "deepseek_v4_random_prompt+net_k10.json")
# LOG_PATH = os.path.join(OUTPUT_DIR, "deepseek_v4_random_prompt+net_k10_errors.log")

# --- DEEPSEEK-V4-FLASH (DeepSeek-V4-Flash, API) | Scenario 3 -- Prompt + Net | No perturbation ---
# OUTPUT_DIR = "/home/siu856622573/Code/RQ_1/deepseek_v4"
# CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "deepseek_v4_no_perturb_prompt+net_scn3_checkpoint.json")
# FINAL_PATH = os.path.join(OUTPUT_DIR, "deepseek_v4_no_perturb_prompt+net_scn3.json")
# LOG_PATH = os.path.join(OUTPUT_DIR, "deepseek_v4_no_perturb_prompt+net_scn3_errors.log")

SAVE_EVERY = 10
RETRY_ATTEMPTS = 2


## 7. JSON parsing (shared, unchanged)

In [9]:
import json

def parse_json(raw):
    raw = raw.strip()
    start = raw.find("{")
    if start == -1:
        return None
    raw = raw[start:]

    decoder = json.JSONDecoder()
    try:
        obj, _end = decoder.raw_decode(raw)
        return obj
    except json.JSONDecodeError as e:
        print(e)
        print(raw)
        return None


## 8. Checkpoint / logging helpers + generation call

`load_checkpoint` / `save_checkpoint` / `log_line` are shared and unchanged. `generate_stage1_response_safe` has 3 possible mechanisms depending on the active model -- exactly one should be active:

1. **Local HF, single-stage** -- Llama-3.1-8B-Instruct, Qwen2.5-14B-Instruct
2. **Local HF, reasoning (think/answer 2-stage)** -- DeepSeek-R1-Distill-Qwen-14B, Qwen3-14B
3. **API (OpenAI-compatible client)** -- GPT-Luna, DeepSeek-V4-Flash

In [10]:
import gc
import json
import os
import time

import torch
from tqdm.auto import tqdm


def load_checkpoint():
    if os.path.exists(CHECKPOINT_PATH):
        with open(CHECKPOINT_PATH, "r") as f:
            loaded = json.load(f)
        # Only trust entries that actually completed successfully -- guards
        # against an old-format checkpoint that might contain error records.
        loaded = {sid: rec for sid, rec in loaded.items() if rec.get("prediction") is not None}
        print(f"Resuming: {len(loaded)} subjects already completed successfully.")
        return loaded
    return {}


def save_checkpoint(results, path=CHECKPOINT_PATH):
    tmp_path = path + ".tmp"
    with open(tmp_path, "w") as f:
        json.dump(results, f, indent=2)
    os.replace(tmp_path, path)  # atomic write


def log_line(msg):
    ts = time.strftime("%Y-%m-%d %H:%M:%S")
    with open(LOG_PATH, "a") as f:
        f.write(f"[{ts}] {msg}\n")


# --- MECHANISM 1: local HF, single-stage (Llama-3.1-8B-Instruct, Qwen2.5-14B-Instruct) ---
def generate_stage1_response_safe(subject_id, brain_graph_text, max_new_tokens=MAX_NEW_TOKENS):
    prompt = build_stage1_prompt(subject_id, brain_graph_text)
    messages = [
        {
            "role": "system",
            "content": (
                "You are a neuroscience expert. Output only one valid JSON object matching the required schema. No markdown, comments, or extra text. Must parse with json.loads()."
            ),
        },
        {"role": "user", "content": prompt},
    ]
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    try:
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        generated = output_ids[0][inputs["input_ids"].shape[1]:]
        text = tokenizer.decode(generated, skip_special_tokens=True).strip()
        return text
    finally:
        # Runs whether generate() succeeded or raised -- keeps memory from
        # accumulating across 800+ sequential calls.
        del inputs
        if "output_ids" in dir():
            del output_ids
        gc.collect()
        torch.cuda.empty_cache()

# --- MECHANISM 2: local HF, reasoning think/answer 2-stage (DeepSeek-R1-Distill-Qwen-14B, Qwen3-14B) ---
# def generate_stage1_response_safe(subject_id, brain_graph_text, max_new_tokens=MAX_NEW_TOKENS):
#     prompt = build_stage1_prompt(subject_id, brain_graph_text)
#     messages = [{"role": "user", "content": prompt}]
#     prompt_text = tokenizer.apply_chat_template(
#         messages, tokenize=False, add_generation_prompt=True,
#     )
#     inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

#     THINK_BUDGET = 500          # tokens allowed for reasoning
#     ANSWER_BUDGET = max_new_tokens - THINK_BUDGET   # remaining for the JSON

#     try:
#         with torch.no_grad():
#             # Stage 1: let it think, but cap it
#             think_ids = model.generate(
#                 **inputs, max_new_tokens=THINK_BUDGET,
#                 do_sample=False, pad_token_id=tokenizer.eos_token_id,
#             )
#             think_text = tokenizer.decode(
#                 think_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
#             )
#             # force-close thinking if it didn't finish on its own
#             if "</think>" not in think_text:
#                 think_text += "\n</think>\n"

#             # Stage 2: continue from forced-closed think, now answer in JSON
#             stage2_prompt = prompt_text + think_text
#             inputs2 = tokenizer(stage2_prompt, return_tensors="pt").to(model.device)
#             answer_ids = model.generate(
#                 **inputs2, max_new_tokens=ANSWER_BUDGET,
#                 do_sample=False, pad_token_id=tokenizer.eos_token_id,
#             )
#             answer_text = tokenizer.decode(
#                 answer_ids[0][inputs2["input_ids"].shape[1]:], skip_special_tokens=True
#             )
#         return think_text + answer_text
#     finally:
#         del inputs
#         gc.collect()
#         torch.cuda.empty_cache()

# --- MECHANISM 3: API, OpenAI-compatible client (GPT-Luna, DeepSeek-V4-Flash) ---
# def generate_stage1_response_safe(subject_id, brain_graph_text, max_new_tokens=MAX_NEW_TOKENS, retries=2):
#     prompt = build_stage1_prompt(subject_id, brain_graph_text)  # unchanged prompt
#     for attempt in range(retries + 1):
#         try:
#             response = client.chat.completions.create(
#                 model=MODEL_ID,
#                 messages=[{"role": "user", "content": prompt}],
#                 max_completion_tokens=max_new_tokens,
#             )
#             return response.choices[0].message.content
#         except Exception:
#             if attempt == retries:
#                 raise
#             time.sleep(2 * (attempt + 1))


def process_one_subject(sid):
    """Returns a result dict on success, or raises the last exception after
    exhausting RETRY_ATTEMPTS."""
    last_exc = None
    for attempt in range(1, RETRY_ATTEMPTS + 1):
        try:
            raw = generate_stage1_response_safe(sid, brain_graph_texts[sid])
            parsed = parse_json(raw)

            gt = int(meta.loc[meta["subject_id"] == sid, "DX_binary"].values[0])
            ground_truth = "Control" if gt == 0 else "ADHD"

            if parsed is None:
                log_line(f"[parse_fail] {sid} (attempt {attempt}): {raw[:300]!r}")
                raise ValueError("Could not parse JSON from model output")

            return {
                "subject_id": sid,
                "ground_truth": ground_truth,
                "prediction": parsed.get("prediction", ""),
                "class_confidence": parsed.get("class_confidence", {}),
                "reasoning": parsed.get("reasoning", ""),
            }

        except Exception as e:
            last_exc = e
            log_line(f"[error] {sid} (attempt {attempt}/{RETRY_ATTEMPTS}): {repr(e)}")
            gc.collect()
            torch.cuda.empty_cache()
            torch.cuda.synchronize()

    raise last_exc


## Sanity check

In [11]:
sid = subject_ids[15]  # or any specific ID, e.g. "1018959"

raw = generate_stage1_response_safe(sid, brain_graph_texts[sid])
print(repr(raw))  # exact raw text, for debugging if parsing fails

parsed = parse_json(raw)

if parsed is None:
    print("\nPARSE FAILED -- raw text above was not valid JSON.")
else:
    print("\nPARSED SUCCESSFULLY:")
    print(f"  prediction:       {parsed.get('prediction')}")
    print(f"  class_confidence: {parsed.get('class_confidence')}")
    print(f"  reasoning:        {parsed.get('reasoning')}")


/home/siu856622573/.conda/envs/py10_roy/lib/python3.10/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


'{\n  "prediction": "ADHD",\n  "class_confidence": {\n    "Control": 0.0,\n    "ADHD": 0.99\n  },\n  "reasoning": "The subject\'s hub regions show a strong deviation from the cohort mean in the Vis, Limbic, and Unassigned networks, with several regions having a positive cohort_z score, indicating above-average connectivity. This pattern is consistent with the altered default mode network (DMN) hub connectivity and reduced frontal-striatal hub strength observed in ADHD."\n}'

PARSED SUCCESSFULLY:
  prediction:       ADHD
  class_confidence: {'Control': 0.0, 'ADHD': 0.99}
  reasoning:        The subject's hub regions show a strong deviation from the cohort mean in the Vis, Limbic, and Unassigned networks, with several regions having a positive cohort_z score, indicating above-average connectivity. This pattern is consistent with the altered default mode network (DMN) hub connectivity and reduced frontal-striatal hub strength observed in ADHD.


In [12]:
# ---------------------------------------------------------------------
# Sanity check -- run this before the main inference loop
# ---------------------------------------------------------------------
assert len(subject_ids) == N_SUBJECTS, f"Expected {N_SUBJECTS} subjects, got {len(subject_ids)}"
assert all(sid in brain_graph_texts for sid in subject_ids), "Missing brain_graph_texts for some subject_ids"
assert meta is not None, "meta not loaded"
assert set(subject_ids) <= set(meta["subject_id"]), "Some subject_ids missing from meta"

gt_counts = meta.loc[meta["subject_id"].isin(subject_ids), "DX_binary"].value_counts()
print(f"Ground truth check: Control={gt_counts.get(0,0)}, ADHD={gt_counts.get(1,0)} "
      f"(expected 488 / 280)")

print(FINAL_PATH)

# one live generation, checked against the schema the prompt actually asks for
_test_sid = subject_ids[0]
_raw = generate_stage1_response_safe(_test_sid, brain_graph_texts[_test_sid])
_parsed = parse_json(_raw)
assert _parsed is not None, f"Sanity-check generation failed to parse JSON:\n{_raw[:500]}"
_expected_keys = {"prediction", "class_confidence", "reasoning"}
_missing = _expected_keys - _parsed.keys()
assert not _missing, f"Parsed JSON missing expected keys: {_missing}\nGot keys: {list(_parsed.keys())}\nRaw output:\n{_raw[:800]}"
assert _parsed["prediction"] in {"Control", "ADHD"}, f"Unexpected prediction value: {_parsed['prediction']}"

print(f"Sanity check passed on subject {_test_sid} -- prediction={_parsed['prediction']}, "
      f"keys OK, {len(subject_ids)} subjects ready, output -> {FINAL_PATH}")


Ground truth check: Control=488, ADHD=280 (expected 488 / 280)
/home/siu856622573/Code/RQ_1/llama-3.1-8b-instruct/llama_clinical_prompt+net_k5.json
Sanity check passed on subject 1018959 -- prediction=Control, keys OK, 768 subjects ready, output -> /home/siu856622573/Code/RQ_1/llama-3.1-8b-instruct/llama_clinical_prompt+net_k5.json


## Inference

In [ ]:
# ---------------------------------------------------------------------
# Resume from checkpoint, figure out what's left to do
# ---------------------------------------------------------------------
results = load_checkpoint()
remaining_ids = [sid for sid in subject_ids if sid not in results]
print(f"Total subjects: {len(subject_ids)} | already done: {len(results)} | remaining: {len(remaining_ids)}")

n_success, n_parse_fail, n_error = 0, 0, 0
start_time = time.time()

with open(LOG_PATH, "a") as log_f:
    pbar = tqdm(remaining_ids, desc="Subjects", unit="subj")
    for sid in pbar:
        try:
            raw = generate_stage1_response_safe(sid, brain_graph_texts[sid])
            parsed = parse_json(raw)

            gt = int(meta.loc[meta["subject_id"] == sid, "DX_binary"].values[0])
            ground_truth = "Control" if gt == 0 else "ADHD"

            if parsed:
                results[sid] = {
                    "subject_id": sid,
                    "ground_truth": ground_truth,
                    "prediction": parsed.get("prediction", ""),
                    "class_confidence": parsed.get("class_confidence", {}),
                    "reasoning": parsed.get("reasoning", ""),
                }
                n_success += 1
            else:
                results[sid] = {
                    "subject_id": sid,
                    "ground_truth": ground_truth,
                    "raw_output": raw,
                    "parsed": None,
                }
                n_parse_fail += 1
                log_f.write(f"[parse_fail] {sid}: {raw[:300]!r}\n")
                log_f.flush()

        except Exception as e:
            n_error += 1
            log_f.write(f"[error] {sid}: {repr(e)}\n")
            log_f.flush()
            gc.collect()
            torch.cuda.empty_cache()

        done = len(results)
        pbar.set_postfix(done=done, ok=n_success, parse_fail=n_parse_fail, err=n_error)

        if done % SAVE_EVERY == 0:
            save_checkpoint(results)

# Final save + promote checkpoint to the "done" filename once everything ran
save_checkpoint(results)
elapsed = time.time() - start_time
print(f"\nFinished this run in {elapsed/60:.1f} min.")
print(f"Total saved: {len(results)}/{len(subject_ids)}  "
      f"(ok={n_success}, parse_fail={n_parse_fail}, error={n_error})")

if len(results) == len(subject_ids):
    os.replace(CHECKPOINT_PATH, FINAL_PATH)
    print(f"All subjects done -> {FINAL_PATH}")
else:
    print(f"Not all subjects done yet -- re-run this cell to resume from {CHECKPOINT_PATH}")


## Results / stats

In [13]:
import json, os

VALID_LABELS = {"ADHD", "Control"}
print(FINAL_PATH)

def print_stats(data):
    n = len(data)

    n_no_pred = sum(1 for r in data.values() if r.get("prediction") is None)
    malformed_entries = [
        (sid, r.get("prediction"))
        for sid, r in data.items()
        if r.get("prediction") is not None and r.get("prediction") not in VALID_LABELS
    ]
    n_malformed_pred = len(malformed_entries)

    scorable = [
        r for r in data.values()
        if r.get("ground_truth") in VALID_LABELS and r.get("prediction") in VALID_LABELS
    ]
    n_scorable = len(scorable)

    n_gt_adhd_all = sum(1 for r in data.values() if r.get("ground_truth") == "ADHD")
    n_gt_control_all = sum(1 for r in data.values() if r.get("ground_truth") == "Control")

    n_pred_adhd = sum(1 for r in scorable if r["prediction"] == "ADHD")
    n_pred_control = sum(1 for r in scorable if r["prediction"] == "Control")

    n_correct = sum(1 for r in scorable if r["prediction"] == r["ground_truth"])

    scorable_gt_adhd = [r for r in scorable if r["ground_truth"] == "ADHD"]
    scorable_gt_control = [r for r in scorable if r["ground_truth"] == "Control"]
    n_adhd_correct = sum(1 for r in scorable_gt_adhd if r["prediction"] == "ADHD")
    n_control_correct = sum(1 for r in scorable_gt_control if r["prediction"] == "Control")

    print(f"Total subjects in file:              {n}")
    print(f"  -- no prediction (parse failure):  {n_no_pred}")
    print(f"  -- malformed prediction (neither ADHD nor Control exactly): {n_malformed_pred}")
    if malformed_entries:
        print(f"     malformed subject_ids and their prediction values:")
        for sid, val in malformed_entries:
            print(f"       {sid}: {val!r}")
    print(f"  -- scorable (valid ground_truth AND valid prediction): {n_scorable}")
    print()
    print(f"Ground truth (all {n} subjects) -- ADHD: {n_gt_adhd_all} | Control: {n_gt_control_all}")
    print(f"Prediction (scorable subjects only, n={n_scorable}) -- ADHD: {n_pred_adhd} | Control: {n_pred_control}")
    print()
    if n_scorable:
        print(f"Accuracy (scorable only): {n_correct}/{n_scorable} = {n_correct / n_scorable:.2%}")
    else:
        print("Accuracy: n/a (no scorable subjects)")

    print()
    print("Correctly predicted, by class (denominator = scorable subjects of that class only):")
    if scorable_gt_adhd:
        print(f"  ADHD    correctly predicted: {n_adhd_correct}/{len(scorable_gt_adhd)} = {n_adhd_correct / len(scorable_gt_adhd):.2%}")
    else:
        print("  ADHD    correctly predicted: n/a")
    if scorable_gt_control:
        print(f"  Control correctly predicted: {n_control_correct}/{len(scorable_gt_control)} = {n_control_correct / len(scorable_gt_control):.2%}")
    else:
        print("  Control correctly predicted: n/a")


if os.path.exists(FINAL_PATH):
    with open(FINAL_PATH) as f:
        data = json.load(f)
    print("RUN COMPLETE.")
    print_stats(data)
else:
    print("No output file found yet -- nothing processed.")


/home/siu856622573/Code/RQ_1/llama-3.1-8b-instruct/llama_clinical_prompt+net_k5.json
RUN COMPLETE.
Total subjects in file:              768
  -- no prediction (parse failure):  0
  -- malformed prediction (neither ADHD nor Control exactly): 0
  -- scorable (valid ground_truth AND valid prediction): 768

Ground truth (all 768 subjects) -- ADHD: 280 | Control: 488
Prediction (scorable subjects only, n=768) -- ADHD: 306 | Control: 462

Accuracy (scorable only): 396/768 = 51.56%

Correctly predicted, by class (denominator = scorable subjects of that class only):
  ADHD    correctly predicted: 107/280 = 38.21%
  Control correctly predicted: 289/488 = 59.22%
